**Regras de associação**

**Algoritmo Apriori**

In [ ]:
!pip install apyori

import pandas as pd
from apyori import apriori

try:
    base = pd.read_csv('./data/compras.csv', sep=',', header=0)

    # Transformar os dados para incluir a presença e a ausência de produtos
    transacoes = []
    todos_produtos = base.columns.tolist()
    for i in range(len(base)):
        transacao = []
        for produto in todos_produtos:
            if base.at[i, produto] == 'Sim':
                transacao.append(produto)
            else:
                transacao.append(f"Nao-{produto}")
        transacoes.append(transacao)

    # Executar o algoritmo Apriori
    regras = apriori(transacoes, min_support=0.3, min_confidence=0.7)
    saida = list(regras)

    # Organizar e processar os resultados
    if saida:
        Antecedente = []
        Consequente = []
        suporte = []
        confianca = []
        lift = []

        for resultado in saida:
            s = resultado.support
            for regra in resultado.ordered_statistics:
                # Ignora regras com antecedentes ou consequentes vazios
                if not regra.items_base or not regra.items_add:
                    continue

                Antecedente.append(list(regra.items_base))
                Consequente.append(list(regra.items_add))
                suporte.append(s)
                confianca.append(regra.confidence)
                lift.append(regra.lift)

        RegrasFinais = pd.DataFrame({
            'Antecedente': Antecedente,
            'Consequente': Consequente,
            'suporte': suporte,
            'confianca': confianca,
            'lift': lift
        })

        # REGRAS DE PRESENÇA
        print("="*60)
        print("Regras de Presença (Quem leva A, também leva B)")
        print("="*60)
        regras_de_presenca = RegrasFinais[
            RegrasFinais['Antecedente'].apply(lambda items: all('Nao-' not in item for item in items)) &
            RegrasFinais['Consequente'].apply(lambda items: all('Nao-' not in item for item in items))
        ]
        display(regras_de_presenca.sort_values(by='confianca', ascending=False))


        #REGRAS DE AUSÊNCIA
        print("\n" + "="*60)
        print("Regras de Ausência (Quem NÃO leva A, leva B)")
        print("="*60)
        # Filtra regras onde o antecedente CONTÉM "Nao-"
        regras_de_ausencia = RegrasFinais[
            RegrasFinais['Antecedente'].apply(lambda items: any('Nao-' in item for item in items)) &
            RegrasFinais['Consequente'].apply(lambda items: all('Nao-' not in item for item in items))
        ]
        # Ordena pelo lift
        display(regras_de_ausencia.sort_values(by='lift', ascending=False))

    else:
        print("Nenhuma regra de associação foi encontrada. Tente usar valores menores para 'min_support' e 'min_confidence'.")

except FileNotFoundError:
    print("Erro: O arquivo 'compras.csv' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")